# Notebook 04: Epidemiological Analytics

## Overview
This notebook calculates advanced epidemiological metrics, trends, and anomaly detection for cholera surveillance.

## Prerequisites
- Notebook 03 completed successfully
- Gold tables: `dim_country`, `dim_date`, `fact_cholera_cases`, `fact_cholera_deaths`

## Inputs
- Gold layer dimensional model

## Outputs
- `gold.epi_analytics_weekly` - Weekly epidemiological summary with trends
- `gold.epi_country_trends` - Country-level time series analysis
- `gold.epi_hotspot_detection` - Anomaly detection results

## Execution Time
~1-2 minutes

In [1]:
# ============================================
# ENVIRONMENT DETECTION & CONFIGURATION
# ============================================

import os
import sys
from pathlib import Path

IS_FABRIC = os.path.exists('/lakehouse/default')

if IS_FABRIC:
    print("🌐 Running in Microsoft Fabric")
    GOLD_TABLE_PATH = "/lakehouse/default/Tables/gold"
else:
    print("💻 Running locally")
    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(project_root / 'src'))
    GOLD_TABLE_PATH = str(project_root / "data" / "gold_tables")

print(f"Gold Path: {GOLD_TABLE_PATH}")

💻 Running locally
Gold Path: D:\Projects\cholera-cdr-mvp\data\gold_tables


In [2]:
# ============================================
# IMPORTS
# ============================================

import pandas as pd
import numpy as np
from datetime import datetime
from typing import Dict, List
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Imports successful")

✅ Imports successful


In [3]:
# ============================================
# DIAGNOSTIC: CHECK DATA STRUCTURE
# ============================================

print("\n🔍 DIAGNOSTIC CHECK")
print("="*60)

print("\n📊 dim_report structure:")
print(f"Columns: {df_dim_report.columns.tolist()}")
print(f"Rows: {len(df_dim_report)}")
print("\nData:")
print(df_dim_report[['report_key', 'report_id', 'epi_year', 'epi_week']])

print("\n📊 fact_cholera_cases structure:")
print(f"Columns: {df_fact_cases.columns.tolist()}")
print(f"Rows: {len(df_fact_cases)}")
print("\nData:")
print(df_fact_cases[['case_key', 'report_key', 'country_key', 'new_cases']])

print("\n📊 Checking report_key overlap:")
print(f"report_keys in dim_report: {sorted(df_dim_report['report_key'].unique())}")
print(f"report_keys in fact_cases: {sorted(df_fact_cases['report_key'].unique())}")

# Check for matching keys
matching_keys = set(df_dim_report['report_key'].unique()) & set(df_fact_cases['report_key'].unique())
print(f"Matching report_keys: {sorted(matching_keys)}")

if len(matching_keys) == 0:
    print("\n❌ PROBLEM: No matching report_keys between tables!")
    print("This is why the merge is failing.")


🔍 DIAGNOSTIC CHECK

📊 dim_report structure:


NameError: name 'df_dim_report' is not defined

In [4]:
# ============================================
# LOAD GOLD DATA
# ============================================

print("\n📂 Loading Gold layer data...\n")

try:
    if IS_FABRIC:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        
        df_dim_country = spark.table("gold.dim_country").toPandas()
        df_dim_date = spark.table("gold.dim_date").toPandas()
        df_dim_report = spark.table("gold.dim_report").toPandas()
        df_fact_cases = spark.table("gold.fact_cholera_cases").toPandas()
        df_fact_deaths = spark.table("gold.fact_cholera_deaths").toPandas()
    else:
        df_dim_country = pd.read_parquet(Path(GOLD_TABLE_PATH) / "dim_country.parquet")
        df_dim_date = pd.read_parquet(Path(GOLD_TABLE_PATH) / "dim_date.parquet")
        df_dim_report = pd.read_parquet(Path(GOLD_TABLE_PATH) / "dim_report.parquet")
        df_fact_cases = pd.read_parquet(Path(GOLD_TABLE_PATH) / "fact_cholera_cases.parquet")
        df_fact_deaths = pd.read_parquet(Path(GOLD_TABLE_PATH) / "fact_cholera_deaths.parquet")
    
    print(f"✅ Loaded {len(df_dim_country)} countries")
    print(f"✅ Loaded {len(df_dim_date)} dates")
    print(f"✅ Loaded {len(df_dim_report)} reports")
    print(f"✅ Loaded {len(df_fact_cases)} case records")
    print(f"✅ Loaded {len(df_fact_deaths)} death records")
    
except Exception as e:
    logger.error(f"Error loading Gold data: {e}")
    raise


📂 Loading Gold layer data...

✅ Loaded 53 countries
✅ Loaded 106 dates
✅ Loaded 3 reports
✅ Loaded 6 case records
✅ Loaded 6 death records


In [5]:
# ============================================
# STEP 4: PREPARE ANALYTICS DATASET
# ============================================

print("\n" + "="*60)
print("🔄 PREPARING ANALYTICS DATASET")
print("="*60 + "\n")

# Start with fact_cholera_cases
df_analytics = df_fact_cases.copy()

print(f"Step 1: Starting with fact_cases")
print(f"  Records: {len(df_analytics)}")
print(f"  Columns: {df_analytics.columns.tolist()}")

# Join with dim_report to get epi_year and epi_week
print(f"\nStep 2: Joining with dim_report...")
print(f"  dim_report has {len(df_dim_report)} reports")
print(f"  Joining on 'report_key'")

df_analytics = df_analytics.merge(
    df_dim_report[['report_key', 'report_id', 'epi_year', 'epi_week']],
    on='report_key',
    how='left'
)

# Verify the merge worked
print(f"  After merge: {len(df_analytics)} records")
print(f"  Columns now: {df_analytics.columns.tolist()}")

# Check for null values
null_check = df_analytics[['report_key', 'epi_year', 'epi_week']].head()
print(f"\nMerge result check:")
print(null_check)

null_year = df_analytics['epi_year'].isna().sum()
null_week = df_analytics['epi_week'].isna().sum()
print(f"  Null epi_year: {null_year}/{len(df_analytics)}")
print(f"  Null epi_week: {null_week}/{len(df_analytics)}")

if null_year > 0 or null_week > 0:
    print("\n❌ ERROR: Merge failed!")
    print("Investigating...")
    print("\nSample of merged data:")
    print(df_analytics[['case_key', 'report_key', 'epi_year', 'epi_week', 'new_cases']].head(10))
    
    # Show which report_keys have nulls
    null_rows = df_analytics[df_analytics['epi_year'].isna()]
    print(f"\nRows with null epi_year:")
    print(null_rows[['case_key', 'report_key', 'new_cases']].head())
else:
    print("  ✅ Merge successful!")

# Join with dim_country to get country details
print(f"\nStep 3: Joining with dim_country...")
df_analytics = df_analytics.merge(
    df_dim_country[['country_key', 'country_code', 'country_name', 'au_region', 'population']],
    on='country_key',
    how='left'
)
print(f"  After merge: {len(df_analytics)} records")

# Join with fact_cholera_deaths to get death data
print(f"\nStep 4: Joining with fact_deaths...")
df_analytics = df_analytics.merge(
    df_fact_deaths[['report_key', 'country_key', 'new_deaths', 'cfr_percent']],
    on=['report_key', 'country_key'],
    how='left',
    suffixes=('', '_deaths')
)
print(f"  After merge: {len(df_analytics)} records")

# Ensure numeric columns
df_analytics['new_cases'] = pd.to_numeric(df_analytics['new_cases'], errors='coerce')
df_analytics['new_deaths'] = pd.to_numeric(df_analytics['new_deaths'], errors='coerce')
df_analytics['epi_year'] = pd.to_numeric(df_analytics['epi_year'], errors='coerce')
df_analytics['epi_week'] = pd.to_numeric(df_analytics['epi_week'], errors='coerce')

# Sort by country and week
df_analytics = df_analytics.sort_values(['country_name', 'epi_year', 'epi_week'])

# Calculate cumulative cases by country
df_analytics['cumulative_cases'] = df_analytics.groupby('country_name')['new_cases'].cumsum()
df_analytics['cumulative_deaths'] = df_analytics.groupby('country_name')['new_deaths'].cumsum()

# Count unique weeks and countries
unique_weeks = df_analytics[df_analytics['epi_year'].notna()][['epi_year', 'epi_week']].drop_duplicates()
unique_countries = df_analytics['country_name'].nunique()

print("\n" + "="*60)
print("✅ ANALYTICS DATASET PREPARED")
print("="*60)
print(f"  Total records: {len(df_analytics)}")
print(f"  Countries: {unique_countries}")
print(f"  Weeks: {len(unique_weeks)}")

if len(unique_weeks) > 0:
    print(f"  Week range: {int(df_analytics['epi_week'].min())}-{int(df_analytics['epi_week'].max())}, {int(df_analytics['epi_year'].mode()[0])}")

print("\n📋 Sample Analytics Data:")
sample_cols = ['country_name', 'epi_year', 'epi_week', 'new_cases', 'new_deaths', 'cfr_percent']
print(df_analytics[sample_cols].head())
print()


🔄 PREPARING ANALYTICS DATASET

Step 1: Starting with fact_cases
  Records: 6
  Columns: ['case_key', 'report_key', 'country_key', 'date_key', 'new_cases', 'cumulative_cases', 'confirmed_cases', 'suspected_cases', 'attack_rate', 'incidence_rate', 'created_at']

Step 2: Joining with dim_report...
  dim_report has 3 reports
  Joining on 'report_key'
  After merge: 6 records
  Columns now: ['case_key', 'report_key', 'country_key', 'date_key', 'new_cases', 'cumulative_cases', 'confirmed_cases', 'suspected_cases', 'attack_rate', 'incidence_rate', 'created_at', 'report_id', 'epi_year', 'epi_week']

Merge result check:
   report_key  epi_year  epi_week
0           1      2025         6
1           1      2025         6
2           2      2025         7
3           2      2025         7
4           3      2025         8
  Null epi_year: 0/6
  Null epi_week: 0/6
  ✅ Merge successful!

Step 3: Joining with dim_country...
  After merge: 6 records

Step 4: Joining with fact_deaths...
  After merge

In [6]:
# ============================================
# CALCULATE EPIDEMIOLOGICAL METRICS
# ============================================

print("\n📊 Calculating epidemiological metrics...\n")

# 1. Week-over-week growth rate
df_analytics['cases_prev_week'] = df_analytics.groupby('country_code')['new_cases'].shift(1)
df_analytics['growth_rate_pct'] = (
    (df_analytics['new_cases'] - df_analytics['cases_prev_week']) / 
    df_analytics['cases_prev_week'].replace(0, np.nan) * 100
).round(2)

# 2. 4-week moving average (cases)
df_analytics['cases_4wk_ma'] = (
    df_analytics.groupby('country_code')['new_cases']
    .rolling(window=4, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
).round(2)

# 3. 4-week moving average (deaths)
df_analytics['deaths_4wk_ma'] = (
    df_analytics.groupby('country_code')['new_deaths']
    .rolling(window=4, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
).round(2)

# 4. Cumulative metrics by country
df_analytics['cumulative_cases_country'] = df_analytics.groupby('country_code')['new_cases'].cumsum()
df_analytics['cumulative_deaths_country'] = df_analytics.groupby('country_code')['new_deaths'].cumsum()

# 5. Attack rate (cumulative cases per 100k population)
df_analytics['attack_rate_cumulative'] = (
    df_analytics['cumulative_cases_country'] / df_analytics['population'] * 100000
).round(2)

print("✅ Calculated growth rates and moving averages")
print(f"   - Average growth rate: {df_analytics['growth_rate_pct'].mean():.2f}%")
print(f"   - Max 4-week MA (cases): {df_analytics['cases_4wk_ma'].max():.0f}")


📊 Calculating epidemiological metrics...

✅ Calculated growth rates and moving averages
   - Average growth rate: 9.32%
   - Max 4-week MA (cases): 398


In [7]:
# ============================================
# ANOMALY DETECTION (Modified Z-Score)
# ============================================

print("\n🚨 Running anomaly detection...\n")

def calculate_modified_zscore(series):
    """
    Calculate Modified Z-Score using Median Absolute Deviation (MAD).
    More robust to outliers than standard Z-score.
    """
    median = series.median()
    mad = np.median(np.abs(series - median))
    
    if mad == 0:
        return pd.Series([0] * len(series), index=series.index)
    
    modified_z = 0.6745 * (series - median) / mad
    return modified_z

# Calculate modified Z-score for cases by country
df_analytics['cases_modified_z'] = (
    df_analytics.groupby('country_code')['new_cases']
    .transform(calculate_modified_zscore)
).round(2)

# Flag anomalies (threshold = 2.5)
ANOMALY_THRESHOLD = 2.5
df_analytics['is_anomaly'] = np.abs(df_analytics['cases_modified_z']) > ANOMALY_THRESHOLD
df_analytics['anomaly_severity'] = pd.cut(
    np.abs(df_analytics['cases_modified_z']),
    bins=[0, 2.5, 3.5, np.inf],
    labels=['NORMAL', 'WARNING', 'CRITICAL']
)

anomaly_count = df_analytics['is_anomaly'].sum()
print(f"✅ Anomaly detection complete")
print(f"   - Anomalies detected: {anomaly_count}")
print(f"   - WARNING: {(df_analytics['anomaly_severity'] == 'WARNING').sum()}")
print(f"   - CRITICAL: {(df_analytics['anomaly_severity'] == 'CRITICAL').sum()}")

if anomaly_count > 0:
    print("\n🚨 Anomalies Detected:")
    anomalies = df_analytics[df_analytics['is_anomaly']][[
        'country_name', 'epi_year', 'epi_week', 'new_cases', 
        'cases_4wk_ma', 'cases_modified_z', 'anomaly_severity'
    ]]
    display(anomalies)


🚨 Running anomaly detection...

✅ Anomaly detection complete
   - Anomalies detected: 1
   - WARNING: 0
   - CRITICAL: 1

🚨 Anomalies Detected:


,country_name,epi_year,epi_week,new_cases,cases_4wk_ma,cases_modified_z,anomaly_severity
3,Zambia,2025,7,372,308.5,42.83,CRITICAL


In [8]:
# ============================================
# CREATE WEEKLY SUMMARY
# ============================================

print("\n📊 Creating weekly epidemiological summary...\n")

# Check if we have the required columns
if 'epi_year' not in df_analytics.columns or 'epi_week' not in df_analytics.columns:
    print("⚠️ ERROR: Missing epi_year or epi_week columns")
    print(f"Available columns: {df_analytics.columns.tolist()}")
    df_weekly_summary = pd.DataFrame()
else:
    # Remove any rows with null epi_year or epi_week
    df_analytics_clean = df_analytics.dropna(subset=['epi_year', 'epi_week'])
    
    if len(df_analytics_clean) == 0:
        print("⚠️ WARNING: All rows have null epi_year/epi_week")
        df_weekly_summary = pd.DataFrame()
    else:
        # Group by epi_year and epi_week
        df_weekly_summary = df_analytics_clean.groupby(['epi_year', 'epi_week']).agg({
            'new_cases': 'sum',
            'new_deaths': 'sum',
            'country_name': 'nunique'
        }).reset_index()
        
        # Rename columns
        df_weekly_summary = df_weekly_summary.rename(columns={
            'country_name': 'affected_countries'
        })
        
        # Calculate weekly CFR
        df_weekly_summary['weekly_cfr'] = (
            df_weekly_summary['new_deaths'] / df_weekly_summary['new_cases'] * 100
        ).fillna(0)
        
        # Count anomalies per week (if anomaly column exists)
        if 'anomaly_severity' in df_analytics_clean.columns:
            anomaly_counts = df_analytics_clean[df_analytics_clean['anomaly_severity'].notna()].groupby(
                ['epi_year', 'epi_week']
            ).size().reset_index(name='anomaly_count')
            
            df_weekly_summary = df_weekly_summary.merge(
                anomaly_counts,
                on=['epi_year', 'epi_week'],
                how='left'
            )
            df_weekly_summary['anomaly_count'] = df_weekly_summary['anomaly_count'].fillna(0).astype(int)
        else:
            df_weekly_summary['anomaly_count'] = 0
        
        # Convert epi_year and epi_week to int
        df_weekly_summary['epi_year'] = df_weekly_summary['epi_year'].astype(int)
        df_weekly_summary['epi_week'] = df_weekly_summary['epi_week'].astype(int)
        
        # Sort by week
        df_weekly_summary = df_weekly_summary.sort_values(['epi_year', 'epi_week'])

print(f"✅ Created weekly summary: {len(df_weekly_summary)} weeks")

if len(df_weekly_summary) > 0:
    print(f"\n📋 Sample Weekly Summary:")
    print(df_weekly_summary.head())
else:
    print("⚠️ Weekly summary is empty - check data joins")

print()


📊 Creating weekly epidemiological summary...

✅ Created weekly summary: 3 weeks

📋 Sample Weekly Summary:
   epi_year  epi_week  new_cases  new_deaths  affected_countries  weekly_cfr  \
0      2025         6        600          16                   2    2.666667   
1      2025         7        781          13                   2    1.664533   
2      2025         8        672          17                   2    2.529762   

   anomaly_count  
0              1  
1              1  
2              2  



In [10]:
# ============================================
# DIAGNOSTIC: Check df_analytics columns
# ============================================

print("\n🔍 CHECKING AVAILABLE COLUMNS")
print("="*60)

print("\nColumns in df_analytics:")
for i, col in enumerate(df_analytics.columns, 1):
    print(f"  {i}. {col}")

print(f"\nTotal columns: {len(df_analytics.columns)}")

print("\nSample data:")
print(df_analytics.head(3))

print("\n" + "="*60 + "\n")


🔍 CHECKING AVAILABLE COLUMNS

Columns in df_analytics:
  1. case_key
  2. report_key
  3. country_key
  4. date_key
  5. new_cases
  6. cumulative_cases
  7. confirmed_cases
  8. suspected_cases
  9. attack_rate
  10. incidence_rate
  11. created_at
  12. report_id
  13. epi_year
  14. epi_week
  15. country_code
  16. country_name
  17. au_region
  18. population
  19. new_deaths
  20. cfr_percent
  21. cumulative_deaths
  22. cases_prev_week
  23. growth_rate_pct
  24. cases_4wk_ma
  25. deaths_4wk_ma
  26. cumulative_cases_country
  27. cumulative_deaths_country
  28. attack_rate_cumulative
  29. cases_modified_z
  30. is_anomaly
  31. anomaly_severity

Total columns: 31

Sample data:
   case_key  report_key  country_key  date_key  new_cases  cumulative_cases  \
1         2           1           52  20250209        245               245   
3         4           2           52  20250216        372               617   
5         6           3           52  20250223        243        

In [11]:
# ============================================
# CREATE COUNTRY TRENDS
# ============================================

print("\n📈 Creating country-level trends...\n")

# First, check what columns we actually have
print("Available columns in df_analytics:")
print(df_analytics.columns.tolist())
print()

# Select only columns that exist
available_cols = [
    'country_key', 'country_code', 'country_name', 'au_region',
    'epi_year', 'epi_week',
    'new_cases', 'new_deaths', 'cumulative_cases', 'cumulative_deaths',
    'cfr_percent', 'incidence_rate'
]

# Add optional columns if they exist
optional_cols = [
    'cases_4wk_ma', 'deaths_4wk_ma', 'growth_rate_pct',
    'attack_rate', 'cases_modified_z', 'anomaly_severity'
]

# Build the column list with only existing columns
trend_cols = []
for col in available_cols:
    if col in df_analytics.columns:
        trend_cols.append(col)
    else:
        print(f"⚠️ Column '{col}' not found, skipping")

for col in optional_cols:
    if col in df_analytics.columns:
        trend_cols.append(col)

print(f"\nUsing columns: {trend_cols}\n")

# Create country trends dataframe
df_country_trends = df_analytics[trend_cols].copy()

# Add trend direction (if growth_rate_pct exists)
if 'growth_rate_pct' in df_country_trends.columns:
    df_country_trends['trend_direction'] = df_country_trends['growth_rate_pct'].apply(
        lambda x: 'INCREASING' if pd.notna(x) and x > 5 
        else ('DECREASING' if pd.notna(x) and x < -5 else 'STABLE')
    )
else:
    # Calculate simple trend direction from new_cases
    df_country_trends['trend_direction'] = 'STABLE'

# Add audit column
df_country_trends['created_at'] = datetime.now()

print(f"✅ Created country trends: {len(df_country_trends)} records")
print(f"   - Countries: {df_country_trends['country_name'].nunique()}")

if 'trend_direction' in df_country_trends.columns:
    print(f"   - Trend distribution:")
    print(df_country_trends['trend_direction'].value_counts())

print()


📈 Creating country-level trends...

Available columns in df_analytics:
['case_key', 'report_key', 'country_key', 'date_key', 'new_cases', 'cumulative_cases', 'confirmed_cases', 'suspected_cases', 'attack_rate', 'incidence_rate', 'created_at', 'report_id', 'epi_year', 'epi_week', 'country_code', 'country_name', 'au_region', 'population', 'new_deaths', 'cfr_percent', 'cumulative_deaths', 'cases_prev_week', 'growth_rate_pct', 'cases_4wk_ma', 'deaths_4wk_ma', 'cumulative_cases_country', 'cumulative_deaths_country', 'attack_rate_cumulative', 'cases_modified_z', 'is_anomaly', 'anomaly_severity']


Using columns: ['country_key', 'country_code', 'country_name', 'au_region', 'epi_year', 'epi_week', 'new_cases', 'new_deaths', 'cumulative_cases', 'cumulative_deaths', 'cfr_percent', 'incidence_rate', 'cases_4wk_ma', 'deaths_4wk_ma', 'growth_rate_pct', 'attack_rate', 'cases_modified_z', 'anomaly_severity']

✅ Created country trends: 6 records
   - Countries: 2
   - Trend distribution:
trend_direct

In [13]:
# ============================================
# CREATE HOTSPOT DETECTION
# ============================================

print("\n🔥 Creating hotspot detection...\n")

# Check if anomaly detection was performed
if 'is_anomaly' not in df_analytics.columns:
    print("⚠️ Warning: No 'is_anomaly' column found. Creating based on anomaly_severity...")
    df_analytics['is_anomaly'] = df_analytics['anomaly_severity'].notna()

# Filter to anomalies only
df_hotspots = df_analytics[df_analytics['is_anomaly']].copy()

if len(df_hotspots) > 0:
    # Select only columns that exist (removed 'date')
    hotspot_cols = [
        'country_key', 'country_code', 'country_name', 'au_region',
        'epi_year', 'epi_week',  # Removed 'date'
        'new_cases', 'cases_4wk_ma', 'cases_modified_z',
        'anomaly_severity', 'incidence_rate'
    ]
    
    df_hotspots = df_hotspots[hotspot_cols].copy()
    
    # Add detection timestamp
    df_hotspots['detection_date'] = datetime.now().date()
    df_hotspots['created_at'] = datetime.now()
    
    print(f"✅ Created hotspot detection: {len(df_hotspots)} hotspot(s)")
    
    print("\n🔥 Hotspot Summary:")
    print(df_hotspots[[
        'country_name', 'epi_year', 'epi_week', 'new_cases', 
        'cases_modified_z', 'anomaly_severity'
    ]].to_string(index=False))
    
else:
    # Create empty DataFrame with schema (removed 'date')
    df_hotspots = pd.DataFrame(columns=[
        'country_key', 'country_code', 'country_name', 'au_region',
        'epi_year', 'epi_week',  # Removed 'date'
        'new_cases', 'cases_4wk_ma', 'cases_modified_z',
        'anomaly_severity', 'incidence_rate',
        'detection_date', 'created_at'
    ])
    print("✅ No hotspots detected (all values within normal range)")

print()


🔥 Creating hotspot detection...

✅ Created hotspot detection: 1 hotspot(s)

🔥 Hotspot Summary:
country_name  epi_year  epi_week  new_cases  cases_modified_z anomaly_severity
      Zambia      2025         7        372             42.83         CRITICAL



In [14]:
# ============================================
# SAVE TO GOLD LAYER
# ============================================

print("\n💾 Saving analytics to Gold layer...\n")

try:
    if IS_FABRIC:
        spark_weekly = spark.createDataFrame(df_weekly_summary)
        spark_trends = spark.createDataFrame(df_country_trends)
        spark_hotspots = spark.createDataFrame(df_hotspots)
        
        spark_weekly.write.format("delta").mode("overwrite").saveAsTable("gold.epi_analytics_weekly")
        spark_trends.write.format("delta").mode("overwrite").saveAsTable("gold.epi_country_trends")
        spark_hotspots.write.format("delta").mode("overwrite").saveAsTable("gold.epi_hotspot_detection")
        
        print("✅ Delta tables created in Fabric Lakehouse")
    else:
        # Convert datetime columns to strings
        for df, name in [
            (df_weekly_summary, 'epi_analytics_weekly'),
            (df_country_trends, 'epi_country_trends'),
            (df_hotspots, 'epi_hotspot_detection')
        ]:
            df_save = df.copy()
            for col in df_save.columns:
                if df_save[col].dtype == 'datetime64[ns]' or 'datetime' in str(df_save[col].dtype):
                    df_save[col] = df_save[col].astype(str)
            
            df_save.to_parquet(
                Path(GOLD_TABLE_PATH) / f"{name}.parquet",
                index=False,
                engine='pyarrow'
            )
            print(f"✅ Saved {name}.parquet ({len(df_save)} rows)")
        
        print(f"\n✅ All files saved to: {GOLD_TABLE_PATH}")
        
except Exception as e:
    logger.error(f"Error saving analytics: {e}")
    raise

print("\n✅ Epidemiological analytics complete!")


💾 Saving analytics to Gold layer...

✅ Saved epi_analytics_weekly.parquet (3 rows)
✅ Saved epi_country_trends.parquet (6 rows)
✅ Saved epi_hotspot_detection.parquet (1 rows)

✅ All files saved to: D:\Projects\cholera-cdr-mvp\data\gold_tables

✅ Epidemiological analytics complete!


## Validation & Testing

In [16]:
# ============================================
# VALIDATION & TESTING
# ============================================

print("\n" + "="*60)
print("🔍 VALIDATION & TESTING")
print("="*60 + "\n")

print("🔍 Running validation checks...\n")

# Test 1: Analytics tables created
if len(df_weekly_summary) > 0:
    print(f"✅ Weekly summary populated: {len(df_weekly_summary)} weeks")
else:
    print(f"⚠️ WARNING: Weekly summary is empty")

if len(df_country_trends) > 0:
    print(f"✅ Country trends populated: {len(df_country_trends)} records")
else:
    print(f"⚠️ WARNING: Country trends is empty")

if len(df_hotspots) > 0:  # Changed from df_hotspot_detection to df_hotspots
    print(f"✅ Hotspot detection populated: {len(df_hotspots)} hotspots")
else:
    print(f"✅ No hotspots detected (this is good!)")

# Test 2: Required columns present
if len(df_analytics) > 0:
    required_cols = ['country_name', 'epi_year', 'epi_week', 'new_cases', 'new_deaths']
    missing_cols = [col for col in required_cols if col not in df_analytics.columns]
    if missing_cols:
        print(f"⚠️ WARNING: Missing columns in analytics: {missing_cols}")
    else:
        print(f"✅ All required columns present in analytics dataset")

# Test 3: Data quality checks
if len(df_analytics) > 0:
    null_epi_year = df_analytics['epi_year'].isna().sum()
    null_epi_week = df_analytics['epi_week'].isna().sum()
    
    if null_epi_year > 0 or null_epi_week > 0:
        print(f"⚠️ WARNING: Null values found - epi_year: {null_epi_year}, epi_week: {null_epi_week}")
    else:
        print(f"✅ No null values in epi_year/epi_week")

# Test 4: Metrics calculated
if len(df_analytics) > 0 and 'cases_4wk_ma' in df_analytics.columns:
    non_null_ma = df_analytics['cases_4wk_ma'].notna().sum()
    print(f"✅ Moving averages calculated for {non_null_ma}/{len(df_analytics)} records")

# Test 5: Anomaly detection
if len(df_analytics) > 0 and 'anomaly_severity' in df_analytics.columns:
    anomaly_count = df_analytics['anomaly_severity'].notna().sum()
    if anomaly_count > 0:
        print(f"✅ Anomalies detected: {anomaly_count}")
        severity_counts = df_analytics['anomaly_severity'].value_counts()
        for severity, count in severity_counts.items():
            print(f"   - {severity}: {count}")
    else:
        print(f"✅ No anomalies detected")

print("\n✅ Validation complete!")
print("="*60)


🔍 VALIDATION & TESTING

🔍 Running validation checks...

✅ Weekly summary populated: 3 weeks
✅ Country trends populated: 6 records
✅ Hotspot detection populated: 1 hotspots
✅ All required columns present in analytics dataset
✅ No null values in epi_year/epi_week
✅ Moving averages calculated for 6/6 records
✅ Anomalies detected: 4
   - NORMAL: 3
   - CRITICAL: 1
   - WARNING: 0

✅ Validation complete!


## Next Steps

1. **Review analytics results** above
2. **Investigate any hotspots** detected
3. **Proceed to Notebook 05** for ML forecasting

## Outputs Created

- `gold.epi_analytics_weekly` - Weekly epidemiological summary with trends
- `gold.epi_country_trends` - Country-level time series with moving averages
- `gold.epi_hotspot_detection` - Anomaly detection results

**Ready for forecasting!** ✅